# Evaluating Models: Why Accuracy Isn't Enough

At the end of the ML overview notebook I left you with a question:

> Your model got about 80% accuracy. Suppose you'd been predicting a rare disease that affects 1 in 100 people. Predicting "healthy" for everybody would score 99%. Would that be a good model?

This notebook answers it, and then builds out the tools you need instead.

**Why now:** your capstone is next week. Somewhere in it you will write a sentence like *"the model achieved X% accuracy"*. This notebook is about making sure that sentence means something, and about the three or four other numbers that should sit beside it.

**What we'll cover:**

1. The accuracy trap, demonstrated properly
2. The confusion matrix, which is where every other metric comes from
3. Precision and recall, and why you usually can't have both
4. Moving the threshold, and choosing which mistake you'd rather make
5. ROC-AUC, one number that doesn't depend on the threshold
6. Cross-validation, because a single train/test split is a lottery

Same Titanic data and the same model as before, so nothing new to learn except the evaluation itself.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report,
)

print("Ready.")

Matplotlib is building the font cache; this may take a moment.


Ready.


---
# Part 1 , The accuracy trap

Let's build the rare disease example and see exactly how bad it gets.

10,000 patients. The disease affects about 1 in 100. Our "model" is one line long: **predict that everybody is healthy.**

In [ ]:
rng = np.random.default_rng(42)

n_patients = 10000
has_disease = (rng.random(n_patients) < 0.01).astype(int)

print("Patients:      ", n_patients)
print("Actually ill:  ", has_disease.sum())
print("Actually healthy:", (has_disease == 0).sum())

In [ ]:
# The laziest model imaginable: everyone is healthy.
lazy_predictions = np.zeros(n_patients, dtype=int)

print("Accuracy: {:.4f}".format(accuracy_score(has_disease, lazy_predictions)))

**98.99% accuracy.**

That would look outstanding in a report. It would beat most models you've built. And it is completely worthless, because it found **zero** of the 101 ill patients.

Every single person who needed treatment was told they were fine.

In [ ]:
print("Ill patients found:  ", (lazy_predictions[has_disease == 1] == 1).sum())
print("Ill patients missed: ", (lazy_predictions[has_disease == 1] == 0).sum())
print()
print("Recall (what fraction of ill patients did we catch?): {:.4f}".format(
    recall_score(has_disease, lazy_predictions, zero_division=0)))

**Recall of 0.0000.** The metric that actually mattered was zero, while the metric everyone quotes was 98.99%.

> ⚠️ **The rule to take away:** accuracy is only meaningful when your classes are roughly balanced. The more imbalanced the data, the more accuracy flatters a useless model. And most interesting real problems are imbalanced, because fraud, disease, equipment failure and customer churn are all rare by nature.

Titanic happens to be reasonably balanced, at 38% survivors, which is why accuracy hasn't misled us so far. Don't count on that in your capstone.

---
# Part 2 , The confusion matrix

The fix is to stop collapsing everything into one number too early.

When a model makes a yes/no prediction there are exactly **four** things that can happen. Two are right and two are wrong, but the two wrongs are *different kinds of wrong*.

|  | Model says NO | Model says YES |
|---|---|---|
| **Actually NO** | True Negative ✓ | False Positive ✗ |
| **Actually YES** | False Negative ✗ | True Positive ✓ |

- **True Negative (TN)** — correctly said no
- **False Positive (FP)** — said yes, was wrong. A *false alarm*.
- **False Negative (FN)** — said no, was wrong. A *miss*.
- **True Positive (TP)** — correctly said yes

The confusion matrix just counts those four. Every other metric in this notebook is arithmetic on these numbers.

In [ ]:
df = pd.read_csv('titanic.csv')

data = df.copy()
data['age'] = data['age'].fillna(data['age'].median())
data['sex_male'] = (data['sex'] == 'male').astype(int)

features = ['pclass', 'sex_male', 'age', 'sibsp', 'parch', 'fare']
X = data[features]
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("Test passengers:", len(y_test))
print("  actually survived:", int(y_test.sum()))
print("  actually died:    ", int((y_test == 0).sum()))
print()
print("Accuracy: {:.4f}".format(accuracy_score(y_test, predictions)))

In [ ]:
cm = confusion_matrix(y_test, predictions)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=['died', 'survived']).plot(ax=ax, cmap='Blues')
plt.title('Confusion matrix')
plt.show()

print(f"True Negatives  (correctly said 'died')     : {tn}")
print(f"False Positives (said survived, actually died): {fp}")
print(f"False Negatives (said died, actually survived): {fn}")
print(f"True Positives  (correctly said 'survived') : {tp}")

**Now the 80.45% has some texture to it.**

Of 179 test passengers the model got 144 right (95 + 49) and 35 wrong. But those 35 split into **15 false positives** and **20 false negatives**, and in a real application those two are rarely equally bad.

Think about what they'd mean in different settings:

| Application | False Positive means | False Negative means |
|---|---|---|
| Cancer screening | Unnecessary worry and further tests | **A missed cancer** |
| Spam filter | **A real email lost to junk** | Some spam in the inbox |
| Fraud detection | A legitimate card declined | Fraud goes through |

For cancer screening you would tolerate a lot of false positives to avoid one false negative. For a spam filter it's the other way round: most people would rather see some spam than lose a real email.

**Accuracy treats those two mistakes as identical. They almost never are.**

---
# Part 3 , Precision and recall

Two numbers that pull the four counts apart, and they answer genuinely different questions.

**Precision — when the model says yes, how often is it right?**

    precision = TP / (TP + FP)

High precision means you can trust a positive prediction. It's what you want when acting on a false alarm is expensive.

**Recall — of all the actual yeses, how many did we catch?**

    recall = TP / (TP + FN)

High recall means few things slip through. It's what you want when *missing* something is expensive.

**F1 score** is the harmonic mean of the two, for when you want a single number that punishes ignoring either one.

In [ ]:
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Precision: {:.4f}".format(precision))
print("Recall:    {:.4f}".format(recall))
print("F1 score:  {:.4f}".format(f1))
print()
print("Check the arithmetic yourself:")
print("  precision = TP / (TP + FP) = {} / ({} + {}) = {:.4f}".format(tp, tp, fp, tp / (tp + fp)))
print("  recall    = TP / (TP + FN) = {} / ({} + {}) = {:.4f}".format(tp, tp, fn, tp / (tp + fn)))

**Precision 0.7656, recall 0.7101.**

In plain English: when the model predicts someone survived it's right about **77%** of the time, and it finds about **71%** of the people who actually survived.

Both numbers are lower than the 80.45% accuracy, which is normal. Accuracy is boosted by all those easy correct "died" predictions. Precision and recall ignore the true negatives entirely and focus on the class you asked about.

scikit-learn will print the whole lot for both classes at once.

In [ ]:
print(classification_report(y_test, predictions,
                            target_names=['died', 'survived'], digits=4))

> 💡 **Read the `support` column.** It's how many test cases were in each class: 110 died, 69 survived. Always check it. A gorgeous F1 score computed on 4 examples is noise, not evidence.

---
# Part 4 , The threshold, and choosing your mistake

Here's something most people don't realise: **the model doesn't output yes or no.** It outputs a probability. The yes/no comes from comparing that probability to a **threshold**, and the default threshold is 0.5.

There is nothing sacred about 0.5. It's just a default, and moving it lets you trade precision against recall deliberately.

In [ ]:
print("What the model actually produces, for the first 8 test passengers:")
for prob, actual in list(zip(probabilities, y_test))[:8]:
    verdict = "survived" if prob >= 0.5 else "died"
    print(f"  P(survived) = {prob:.4f}  ->  predict {verdict:8}  (actually {'survived' if actual else 'died'})")

In [ ]:
print(f"{'threshold':>10} {'accuracy':>9} {'precision':>10} {'recall':>8} {'F1':>8} {'misses':>8} {'alarms':>8}")
print("-" * 66)

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds_t = (probabilities >= threshold).astype(int)
    c = confusion_matrix(y_test, preds_t)
    _, fp_t, fn_t, _ = c.ravel()
    print(f"{threshold:>10.1f} {accuracy_score(y_test, preds_t):>9.4f} "
          f"{precision_score(y_test, preds_t, zero_division=0):>10.4f} "
          f"{recall_score(y_test, preds_t):>8.4f} "
          f"{f1_score(y_test, preds_t):>8.4f} {fn_t:>8} {fp_t:>8}")

**Read down the precision and recall columns. They move in opposite directions.**

- At **threshold 0.3** the model is generous with "survived". Recall climbs to 0.7971, precision falls to 0.6707. Only 14 misses but 27 false alarms.
- At **threshold 0.7** it's stingy. Precision reaches **0.9394**, so when it says survived it's almost always right. But recall collapses to **0.4493**, and it now misses 38 people.

That's the **precision-recall trade-off**, and there is no setting that fixes both. You are choosing which kind of mistake you would rather make.

**And that choice is a business decision, not a statistical one.** The right threshold depends entirely on what each mistake costs, which is a conversation with whoever owns the problem, not something you can read off a chart.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 50)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    preds_t = (probabilities >= t).astype(int)
    precisions.append(precision_score(y_test, preds_t, zero_division=0))
    recalls.append(recall_score(y_test, preds_t))
    f1s.append(f1_score(y_test, preds_t, zero_division=0))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions, label='precision', lw=2)
plt.plot(thresholds, recalls, label='recall', lw=2)
plt.plot(thresholds, f1s, label='F1', lw=2, ls='--')
plt.axvline(0.5, color='grey', ls=':', label='default threshold')
plt.xlabel('threshold')
plt.ylabel('score')
plt.title('You can have precision or recall, not both')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The two solid lines cross. That crossing point is roughly where F1 peaks, and it's a reasonable default when you have no information about costs. But if you *do* know the costs, ignore the crossing point and pick deliberately.

---
# Part 5 , ROC-AUC: one number, every threshold

Every metric so far depended on picking a threshold. That's awkward when you want to compare two models, because a fair comparison would need the same threshold, and the best threshold might differ between them.

**ROC-AUC** solves this by evaluating across *all* thresholds at once.

The ROC curve plots the true positive rate against the false positive rate as the threshold sweeps from 1 down to 0. **AUC** is the area underneath it.

- **1.0** — perfect
- **0.5** — no better than a coin flip
- **below 0.5** — worse than guessing, which usually means a bug

There's a lovely plain-English reading: **AUC is the probability that the model gives a randomly chosen survivor a higher score than a randomly chosen non-survivor.**

In [ ]:
auc = roc_auc_score(y_test, probabilities)
fpr, tpr, _ = roc_curve(y_test, probabilities)

plt.figure(figsize=(6, 5.5))
plt.plot(fpr, tpr, lw=2, label=f'our model (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='random guessing (AUC = 0.5)')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

print("ROC-AUC: {:.4f}".format(auc))

**AUC of 0.8518**, comfortably above the 0.5 diagonal.

So: pick a random survivor and a random non-survivor, and about **85%** of the time the model assigns a higher survival probability to the one who actually survived. That holds regardless of where you put the threshold.

> 💡 **When to quote which.** Use **AUC** to compare models. Use **precision and recall at your chosen threshold** to describe what a deployed model will actually do. AUC is a good screening number and a poor description of real-world behaviour, because in production you always have to pick a threshold in the end.

> ⚠️ **AUC has its own blind spot.** On heavily imbalanced data it can look reassuringly high while the model is still useless in practice. For rare events, the precision-recall curve is more honest.

---
# Part 6 , One split is a lottery

Now the part that matters most for your capstone.

Every number so far came from **one** train/test split with `random_state=42`. What if we'd picked a different split?

In [ ]:
single_split_scores = []

for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    score = accuracy_score(yte, m.predict(Xte))
    single_split_scores.append(score)
    print(f"  random_state={seed}: {score:.4f}")

print()
print("Lowest:  {:.4f}".format(min(single_split_scores)))
print("Highest: {:.4f}".format(max(single_split_scores)))
print("Spread:  {:.1f} percentage points".format((max(single_split_scores) - min(single_split_scores)) * 100))

**The same model, the same data, and the score ranges from 75.98% to 84.36%.**

An 8.4 point spread, caused by nothing but which passengers happened to land in the test set.

Think about what that means. If you tried two models and one scored 84% while the other scored 76%, you might reasonably conclude the first was much better. But you could get that gap from **the identical model** just by changing the split. Any comparison based on a single split is on very shaky ground.

### Cross-validation

The fix: split the data into 5 parts, train on 4 and test on the 1 left out, and rotate so every part gets a turn as the test set. You get five scores instead of one, and the spread tells you how much to trust them.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv)

print("Fold scores:", np.round(cv_scores, 4).tolist())
print()
print("Mean:               {:.4f}".format(cv_scores.mean()))
print("Standard deviation: {:.4f}".format(cv_scores.std()))
print()
print("Report it like this: {:.3f} +/- {:.3f}".format(cv_scores.mean(), cv_scores.std()))

**0.788 ± 0.011.**

That's a far more honest statement than "80.45%". It says: this model gets about 79%, and it's consistent, because the folds only vary by about a percentage point.

> 💡 **`StratifiedKFold` rather than plain `KFold`.** Stratified keeps the same survived/died proportion in every fold. Without it you can get a fold with an unrepresentative mix, which adds noise for no reason. Always stratify for classification.

### The standard deviation tells you something the mean doesn't

Compare our logistic regression against the unlimited decision tree, the one that overfitted badly in the algorithms notebook.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)

logistic_cv = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv)
tree_cv = cross_val_score(tree, X, y, cv=cv)

print("Logistic regression")
print("  folds:", np.round(logistic_cv, 4).tolist())
print("  {:.4f} +/- {:.4f}".format(logistic_cv.mean(), logistic_cv.std()))
print()
print("Decision tree (no depth limit)")
print("  folds:", np.round(tree_cv, 4).tolist())
print("  {:.4f} +/- {:.4f}".format(tree_cv.mean(), tree_cv.std()))

**The tree has a slightly higher mean, 0.7924 against 0.7879, and nearly four times the standard deviation: 0.0424 against 0.0110.**

Look at the tree's individual folds: one scored 0.8596 and another 0.7416. That's a 12 point swing depending on which passengers it happened to be tested on. The logistic regression sits between 0.7697 and 0.8034 throughout.

**The tree isn't better, it's less reliable.** Its higher mean comes from one lucky fold. If you deployed it you'd have very little idea which end of that range you'd get.

> ⚠️ **For your capstone:** report the mean *and* the standard deviation. A model with a slightly lower mean and much smaller spread is usually the better choice, and being able to say why will read very well.

---
# Part 7 , A checklist for the capstone

When you evaluate a model next week, work through this:

**1. Is the data balanced?**
Check `y.value_counts()`. If one class is under about 20%, accuracy alone is misleading and you should lead with precision, recall and F1.

**2. Show the confusion matrix.**
It's four numbers and it explains every other metric you quote. It's also the easiest thing for a non-technical reader to follow.

**3. Quote precision and recall, not just accuracy.**
And say which one matters more for your problem, and why. That sentence is worth more than the numbers.

**4. Say what your errors mean.**
"20 false negatives" is a statistic. "The model missed 20 passengers who actually survived" is a finding. Translate.

**5. Use cross-validation, report mean ± std.**
A single split can swing 8 points on this dataset. Anyone assessing your work will know that.

**6. Always compare against a baseline.**
Predicting the majority class scores 61.6% here, and one line of logic about sex scores 78.7%. If your model doesn't clearly beat both, say so honestly. Noticing that is a much better look than not noticing.

---

## The short version

| Metric | Answers | Reach for it when |
|---|---|---|
| **Accuracy** | How many did we get right? | Classes are balanced |
| **Precision** | When it says yes, is it right? | False alarms are costly |
| **Recall** | Of all the yeses, how many found? | Misses are costly |
| **F1** | Balance of the two | You need one number |
| **ROC-AUC** | How good at ranking, at any threshold? | Comparing models |
| **CV mean ± std** | How good, and how consistent? | Always |

> **The one sentence to remember:** accuracy is a summary, and summaries hide things. The confusion matrix is where the truth is.